In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Operator-Rückpropagation (OBP) zur Schätzung von Erwartungswerten

*Geschätzter Verbrauch: 16 Minuten auf einem Eagle-r3-Prozessor (HINWEIS: Dies ist nur eine Schätzung. Deine tatsächliche Laufzeit kann abweichen.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Hintergrund
Operator-Rückpropagation ist eine Technik, bei der Operationen vom Ende eines Quantenschaltkreises in die gemessene Observable absorbiert werden, wodurch die Tiefe des Schaltkreises im Allgemeinen auf Kosten zusätzlicher Terme in der Observable reduziert wird. Das Ziel ist es, so viel wie möglich des Schaltkreises zurückzupropagieren, ohne dass die Observable zu stark anwächst. Eine Qiskit-basierte Implementierung ist im OBP-Qiskit-Addon verfügbar; weitere Details findest du in der entsprechenden [Dokumentation](/guides/qiskit-addons-obp) mit einem [einfachen Beispiel](/guides/qiskit-addons-obp-get-started) für den Einstieg.

Betrachte einen Beispielschaltkreis, für den eine Observable $O = \sum_P c_P P$ gemessen werden soll, wobei $P$ Pauli-Operatoren und $c_P$ Koeffizienten sind. Bezeichnen wir den Schaltkreis als eine einzelne unitäre Transformation $U$, die logisch in $U = U_C U_Q$ partitioniert werden kann, wie in der folgenden Abbildung dargestellt.

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Die Operator-Rückpropagation absorbiert die unitäre Transformation $U_C$ in die Observable, indem sie diese als $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$ entwickelt. Mit anderen Worten: Ein Teil der Berechnung wird klassisch über die Entwicklung der Observable von $O$ zu $O'$ durchgeführt. Das ursprüngliche Problem kann nun als Messung der Observable $O'$ für den neuen Schaltkreis mit geringerer Tiefe umformuliert werden, dessen unitäre Transformation $U_Q$ ist.

Die unitäre Transformation $U_C$ wird als eine Anzahl von Schichten dargestellt: $U_C = U_S U_{S-1}...U_2U_1$. Es gibt mehrere Möglichkeiten, eine Schicht zu definieren. Zum Beispiel kann im obigen Beispielschaltkreis jede Schicht von $R_{zz}$- und jede Schicht von $R_x$-Gattern als individuelle Schicht betrachtet werden. Die Rückpropagation beinhaltet die klassische Berechnung von $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$. Jede Schicht $U_s$ kann als $U_s = exp(\frac{-i\theta_s P_s}{2})$ dargestellt werden, wobei $P_s$ ein $n$-Qubit-Pauli-Operator und $\theta_s$ ein Skalar ist. Es lässt sich leicht verifizieren, dass

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Wenn im obigen Beispiel ${P,P_s} = 0$ gilt, müssen wir zwei Quantenschaltkreise anstelle von einem ausführen, um den Erwartungswert zu berechnen. Daher kann die Rückpropagation die Anzahl der Terme in der Observable erhöhen, was zu einer höheren Anzahl von Schaltkreisausführungen führt. Eine Möglichkeit, eine tiefere Rückpropagation in den Schaltkreis zu ermöglichen und gleichzeitig ein zu starkes Anwachsen des Operators zu verhindern, besteht darin, Terme mit kleinen Koeffizienten zu abschneiden, anstatt sie dem Operator hinzuzufügen. Beispielsweise kann man im obigen Beispiel den Term mit $P_sP$ abschneiden, sofern $\theta_s$ hinreichend klein ist. Das Abschneiden von Termen kann zu weniger auszuführenden Quantenschaltkreisen führen, verursacht jedoch einen Fehler bei der endgültigen Erwartungswertberechnung, der proportional zur Größe der Koeffizienten der abgeschnittenen Terme ist.

Dieses Tutorial implementiert ein [Qiskit-Pattern](/guides/intro-to-patterns) zur Simulation der Quantendynamik einer Heisenberg-Spinkette unter Verwendung von <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>.
## Voraussetzungen
Bevor du mit diesem Tutorial beginnst, stelle sicher, dass du Folgendes installiert hast:

- Qiskit SDK v1.2 oder höher (`pip install qiskit`)
- Qiskit Runtime v0.28 oder höher (`pip install qiskit-ibm-runtime`)
- OBP-Qiskit-Addon (`pip install qiskit-addon-obp`)
- Qiskit-Addon-Utilities (`pip install qiskit-addon-utils`)
## Einrichtung

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## Teil I: Heisenberg-Spinkette im kleinen Maßstab
### Schritt 1: Klassische Eingaben auf ein Quantenproblem abbilden
#### Die Zeitentwicklung eines quantenmechanischen Heisenberg-Modells auf ein Quantenexperiment abbilden.
Das Paket [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) bietet einige wiederverwendbare Funktionalitäten für verschiedene Zwecke.

Sein Modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) stellt Funktionen zur Erzeugung von Heisenberg-artigen Hamiltonoperatoren auf einem gegebenen Konnektivitätsgraphen bereit.
Dieser Graph kann entweder ein [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) oder eine [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap) sein, was die Verwendung in Qiskit-zentrierten Arbeitsabläufen erleichtert.

Im Folgenden erzeugen wir eine lineare Ketten-`CouplingMap` mit 10 Qubits.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

Als Nächstes erzeugen wir einen Pauli-Operator, der einen Heisenberg-XYZ-Hamiltonoperator modelliert.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Dabei ist $G(V,E)$ der Graph der bereitgestellten Coupling Map.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


Aus dem Qubit-Operator können wir einen Quantenschaltkreis erzeugen, der dessen Zeitentwicklung modelliert.
Auch hier hilft uns das Modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) mit einer praktischen Funktion genau dafür:

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### Schritt 2: Problem für die Ausführung auf Quantenhardware optimieren
#### Schaltkreis-Schichten für die Rückpropagation erstellen
Beachte, dass die Funktion ``backpropagate`` jeweils ganze Schaltkreis-Schichten zurückpropagiert. Daher kann die Wahl der Aufteilung einen Einfluss darauf haben, wie gut die Rückpropagation für ein gegebenes Problem funktioniert. Hier gruppieren wir Gatter gleichen Typs in Schichten mithilfe der Funktion [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types).

Für eine ausführlichere Diskussion zur Schaltkreisaufteilung lies diese [Anleitung](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) des Pakets [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils).

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Begrenzung des Operatorwachstums während der Rückpropagation
Während der Rückpropagation nähert sich die Anzahl der Terme im Operator im Allgemeinen schnell $4^N$ an, wobei $N$ die Anzahl der Qubits ist. Wenn zwei Terme im Operator nicht qubitweise kommutieren, benötigen wir separate Schaltkreise, um die entsprechenden Erwartungswerte zu erhalten. Wenn wir beispielsweise eine 2-Qubit-Observable $O = 0.1 XX + 0.3 IZ - 0.5 IX$ haben, reicht, da $[XX,IX] = 0$ gilt, eine Messung in einer einzigen Basis aus, um die Erwartungswerte für diese beiden Terme zu berechnen. Allerdings antikommutiert $IZ$ mit den beiden anderen Termen. Daher benötigen wir eine separate Basismessung, um den Erwartungswert von $IZ$ zu berechnen. Mit anderen Worten: Wir benötigen zwei statt einen Schaltkreis, um $\langle O \rangle$ zu berechnen. Mit steigender Anzahl der Terme im Operator besteht die Möglichkeit, dass auch die erforderliche Anzahl der Schaltkreisausführungen zunimmt.

Die Größe des Operators kann durch Angabe des ``operator_budget``-Parameters der Funktion ``backpropagate`` begrenzt werden, der eine [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget)-Instanz akzeptiert.

Um die Menge der zusätzlich zugewiesenen Ressourcen (Zeit) zu kontrollieren, beschränken wir die maximale Anzahl der qubitweise kommutierenden Pauli-Gruppen, die die zurückpropagierte Observable haben darf. Hier legen wir fest, dass die Rückpropagation gestoppt werden soll, wenn die Anzahl der qubitweise kommutierenden Pauli-Gruppen im Operator 8 überschreitet.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Beachte, dass wir durch die Zuweisung eines Fehlers von `5e-3` pro Schicht für die Abschneidung eine weitere Schicht aus dem Schaltkreis entfernen können, während wir innerhalb des ursprünglichen Budgets von acht kommutierenden Pauli-Gruppen in der Observable bleiben. Standardmäßig verwendet `backpropagate` die L1-Norm der abgeschnittenen Koeffizienten, um den Gesamtfehler aus der Abschneidung zu begrenzen. Für weitere Optionen lies die [Anleitung zur Angabe der p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

In diesem speziellen Beispiel, in dem wir sieben Schichten zurückpropagiert haben, sollte der gesamte Abschneidungsfehler ``(5e-3 Fehler/Schicht) * (7 Schichten) = 3,5e-2`` nicht überschreiten.
Für eine weiterführende Diskussion zur Verteilung eines Fehlerbudgets über deine Schichten lies [diese Anleitung](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>